# DreamerV3

> DreamerV3 (Hafner et al., 2023) on MultiGrid, from r2dreamer's PyTorch implementation (vendored): one world model and actor-critic shared by the agents, trained online.

In [ ]:
#| default_exp wm.dreamer.core

In [ ]:
#| hide
from nbdev.showdoc import *

[DreamerV3](https://arxiv.org/abs/2301.04104) learns a world model (an RSSM with discrete latents,
trained on replayed sequences) and an actor-critic in its imagination. This module runs the
PyTorch DreamerV3 of [r2dreamer](https://github.com/NM512/r2dreamer) (vendored in
`stable_marl.wm.dreamer.r2dreamer`, see its README; `rep_loss='dreamer'`, or R2-Dreamer's
`'r2dreamer'`) on MultiGrid envs:

* **one row per agent**: every agent of every env is a row of the batch, with the same weights
  (independent Dreamers with parameter sharing, CoDreamer's "IDreamer"); rows are ordered by env, then
  agent, so a subclass can regroup the agents of an env (see `stable_marl.wm.codreamer`);
* **streams**: each env plays on, reset when every agent is done; each agent's steps are recorded
  as r2dreamer's transitions (`is_first`, `is_last`, `is_terminal`, `reward`, the action taken, and the
  latent state when acting), and a `mask` marks the steps of its episode: after it reached its goal,
  while the others act on, its steps are padding (zero observations, masked out of the losses);
* **replay**: `SequenceReplay` samples windows of `batch_length + 1` steps of every agent of an env
  (the first step gives the starting latent state), with r2dreamer's `Buffer` interface;
* **training schedule**: r2dreamer's `train_ratio`, the replayed steps per collected step.

The observations are the features of the other baselines (`commnet_features`: each agent's
symbolic view `image`, its `direction`, and its offset to its goal), symlog-encoded by the MLP
encoder. With `image_key` (e.g. `'pixels'`, each agent's RGB view), that view is an image for
r2dreamer's CNN encoder (and CNN decoder with `rep_loss='dreamer'`), resized to `image_size`
(64: r2dreamer's size; the decoder needs a multiple of 16), and `obs_keys` are the other,
vector observations (e.g. `['direction']`).

In [ ]:
#| export
from __future__ import annotations

import copy
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn.functional as F
from fastcore.utils import patch
from omegaconf import DictConfig, OmegaConf

from stable_marl.wm.commnet.core import commnet_features
from stable_marl.wm.dreamer import r2dreamer as _r2dreamer
from stable_marl.wm.dreamer.r2dreamer.dreamer import Dreamer

### Configuration

`dreamer_config` reads r2dreamer's own model configuration (`configs/model/_base_.yaml` and a size,
e.g. `size12M.yaml`) and applies `overrides` (e.g. a smaller `deter`, `hidden`, `units` for a CPU);
the encoder and decoder take every vector observation.

In [ ]:
#| export
def dreamer_config(
    size: str = '12M', # r2dreamer's model size: 12M, 25M, 50M, 100M, 200M, 400M
    rep_loss: str = 'dreamer', # 'dreamer' (DreamerV3), 'r2dreamer', 'infonce' or 'dreamerpro' (images only)
    device: str = 'cpu',
    **overrides, # Model settings, e.g. deter=256, hidden=64, units=64, act_entropy=3e-4, kl_free=1.0
) -> DictConfig:
    "The configuration of r2dreamer's `Dreamer` (its `model` node)."
    folder = Path(_r2dreamer.__file__).parent / 'configs' / 'model'
    model = OmegaConf.merge(OmegaConf.load(folder / '_base_.yaml'), OmegaConf.load(folder / f'size{size}.yaml'))
    model.pop('defaults', None)
    model.update(rep_loss=rep_loss, compile=False, **overrides)
    keys = {'mlp_keys': '.*', 'cnn_keys': '$^'}
    root = OmegaConf.create({'model': model, 'device': device, 'env': {'encoder': keys, 'decoder': keys}})
    OmegaConf.resolve(root)
    return root.model

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
cfg = dreamer_config(deter=256, hidden=64, units=64)
test_eq((cfg.rssm.deter, cfg.rssm.hidden, cfg.actor.units, cfg.rep_loss, cfg.compile), (256, 64, 64, 'dreamer', False))
test_eq(cfg.rssm.device, 'cpu')

### Replay

`SequenceReplay` keeps the last `capacity` steps of each env's stream, every step holding all its
agents. `sample` takes `batch_size` windows of `batch_length + 1` consecutive steps (at random envs
and times) and returns r2dreamer's batch: `data` (rows `(batch_size * num_agents, batch_length, ...)`,
ordered by window then agent), with each step's action being the one taken before it (the action
leading to it), and `initial`, the latent state recorded at the window's first step.

In [ ]:
#| export
class SequenceReplay:
    "Streams of transitions of every env (all their agents per step), sampled as r2dreamer's `Buffer`."
    def __init__(
        self,
        num_envs: int,
        batch_size: int = 16, # Windows per batch (each gives a row per agent)
        batch_length: int = 64, # Steps per window (plus one for the starting latent state)
        capacity: int = 100_000, # Steps kept per env
        device: str | torch.device = 'cpu', # Where batches go
    ):
        self.batch_size, self.batch_length, self.capacity = batch_size, batch_length, capacity
        self.device = torch.device(device)
        self.streams: list[dict[str, list]] = [dict() for _ in range(num_envs)]
        self.rng = np.random.default_rng()

    def add(self, transitions: dict[str, torch.Tensor]):
        "Add one step of every env: tensors (num_envs, num_agents, ...)."
        for e, stream in enumerate(self.streams):
            for k, v in transitions.items():
                stream.setdefault(k, []).append(v[e].detach().cpu())
                if len(stream[k]) > self.capacity:
                    del stream[k][0]

    def __len__(self) -> int:
        "Steps that can start a window."
        return sum(max(0, len(s.get('action', ())) - self.batch_length) for s in self.streams)

    def sample(self, prefer_online: bool = True):
        "`(data, index, initial)` as r2dreamer's `Buffer.sample` (`index` unused: latents are not refreshed)."
        L = self.batch_length + 1
        starts = [(e, len(s['action']) - L) for e, s in enumerate(self.streams) if len(s.get('action', ())) >= L]
        if not starts:
            raise RuntimeError(f"no stream has {L} steps yet")
        picks = self.rng.integers(len(starts), size=self.batch_size)
        windows = []
        for p in picks:
            e, last = starts[p]
            t = int(self.rng.integers(last + 1))
            windows.append({k: torch.stack(v[t:t + L]) for k, v in self.streams[e].items()})   # (L, N, ...)
        batch = {k: torch.stack([w[k] for w in windows]).transpose(1, 2) for k in windows[0]}   # (W, N, L, ...)
        batch = {k: v.flatten(0, 1).to(self.device) for k, v in batch.items()}                  # (W * N, L, ...)
        initial = (batch.pop('stoch')[:, 0], batch.pop('deter')[:, 0])
        data = {k: v[:, 1:] for k, v in batch.items()}
        data['action'] = batch['action'][:, :-1]                                                # the action before each step
        return data, None, initial

    def update(self, index, stoch, deter):
        "r2dreamer's latent refresh: not used."

In [ ]:
#| hide
rep = SequenceReplay(num_envs=2, batch_size=3, batch_length=4, capacity=10)
for t in range(12):
    rep.add({'obs': torch.full((2, 2, 1), float(t)), 'action': torch.full((2, 2, 4), float(t)),
             'stoch': torch.zeros(2, 2, 3, 2), 'deter': torch.full((2, 2, 5), float(t)), 'is_first': torch.zeros(2, 2, 1, dtype=torch.bool)})
test_eq(len(rep.streams[0]['obs']), 10)                                   # the last `capacity` steps
data, _, (stoch, deter) = rep.sample()
test_eq(data['obs'].shape, (6, 4, 1)); test_eq(deter.shape, (6, 5))     # 3 windows x 2 agents
test_eq(data['obs'][:, 0, 0], deter[:, 0] + 1)                           # the window's first step gives the latent
test_eq(data['action'][:, :, 0], data['obs'][:, :, 0] - 1)               # each step's action is the one before it

### The agents

`DreamerAgents` holds r2dreamer's `Dreamer` and runs it with one row per agent. `transitions`
turns columns of one step (`(E, N, ...)`) into the transitions of every agent: observations
(zeros for agents whose episode ended), `reward` (its own, or the team's with `team_reward`),
`is_first`, `is_last` (its episode ends: its goal, or the env's end), `is_terminal` (its goal),
`mask`, and its `position` (for communication, see CoDreamer). As the model of a
`FeedForwardPolicy`, `get_action` keeps every agent's latent state, restarted when its env resets
(`_needs_flush`), and picks the most likely action (`temperature` 0) or samples one.

In [ ]:
#| export
class DreamerAgents(torch.nn.Module):
    "r2dreamer's DreamerV3, one row per agent of each env (weights shared by the agents)."
    def __init__(
        self,
        config: DictConfig, # See `dreamer_config`
        obs_dim: int, # Vector features per agent (see `commnet_features` of `obs_keys`, and `goal`; 0: none)
        num_agents: int,
        num_actions: int = 4,
        obs_keys: Sequence[str] = ('image', 'direction'), # Each agent's vector observations
        goal: bool = True, # Observe the offset to the goal cell
        grid_size: int = 15,
        team_reward: bool = False, # Every agent's reward is the sum of the agents' rewards
        image_key: str | Sequence[str] | None = None, # Image observations for the CNNs (e.g. 'pixels', each agent's RGB view; several, e.g. ['pixels', 'goal'], stacked on the channels)
        image_size: int = 64, # Side the image is resized to (the CNN decoder needs a multiple of 16)
    ):
        super().__init__()
        self.config = dict(config=OmegaConf.to_container(config), obs_dim=obs_dim, num_agents=num_agents,
                           num_actions=num_actions, obs_keys=list(obs_keys), goal=goal, grid_size=grid_size,
                           team_reward=team_reward, image_key=image_key, image_size=image_size)
        self.num_agents, self.num_actions, self.obs_dim = num_agents, num_actions, obs_dim
        self.obs_keys, self.goal, self.grid_size, self.team_reward = tuple(obs_keys), goal, grid_size, team_reward
        self.image_key, self.image_size = image_key, image_size
        self.image_keys = [image_key] if isinstance(image_key, str) else list(image_key or [])
        spaces = {'obs': SimpleNamespace(shape=(obs_dim,))} if obs_dim else {}
        config = copy.deepcopy(config)
        if self.image_keys:                                         # r2dreamer's CNN encoder and decoder take it
            spaces['image'] = SimpleNamespace(shape=(image_size, image_size, 3 * len(self.image_keys)))
            config.encoder.cnn_keys = config.decoder.cnn_keys = '^image$'
        if not spaces:
            raise ValueError("no observations: give obs_keys (or goal) and/or an image_key")
        act_space = SimpleNamespace(n=num_actions, discrete=True, shape=(num_actions,))
        self.agent = self._build(config, SimpleNamespace(spaces=spaces), act_space)
        self.agent.to(self.agent.device)                            # r2dreamer builds on the CPU: its .to() moves the networks and re-shares the frozen copies
        self.temperature = 0.0
        self.checkpoint: dict = {}
        self._state: dict | None = None

    def _build(self, config: DictConfig, obs_space, act_space) -> Dreamer:
        "The model (a subclass changes it, e.g. CoDreamer)."
        return Dreamer(config, obs_space, act_space)

    @property
    def device(self) -> torch.device:
        return self.agent.device

    def on_reset(self, envs: Sequence[int], seeds: Sequence[int | None] | None = None):
        "Called when `envs` start a new episode (with their `seeds`); a subclass can reset what it keeps per env."

    def info_keys(self) -> list[str]:
        keys = [*self.obs_keys, 'terminated', 'position'] + self.image_keys
        return list(dict.fromkeys(keys + (['goal_position'] if self.goal else [])))

    def image(self, columns: dict[str, np.ndarray]) -> torch.Tensor:
        "Each agent's images (E, N, image_size, image_size, 3 per image key), uint8, stacked on the channels."
        out = []
        for key in self.image_keys:
            x = torch.as_tensor(np.asarray(columns[key]))
            lead = x.shape[:-3]
            if x.shape[-3:-1] != (self.image_size, self.image_size):
                x = F.interpolate(x.reshape(-1, *x.shape[-3:]).permute(0, 3, 1, 2).float(), size=(self.image_size,) * 2,
                                  mode='bilinear', antialias=True, align_corners=False)
                x = x.round().clamp(0, 255).to(torch.uint8).permute(0, 2, 3, 1)
            out.append(x.reshape(*lead, self.image_size, self.image_size, 3))
        return torch.cat(out, -1)

    def transitions(
        self,
        columns: dict[str, np.ndarray], # One step (E, N, ...) of the info dict
        reward: np.ndarray, # (E, N) the rewards of the step that led here (0 after a reset)
        is_first: np.ndarray, # (E,) envs that just reset
        env_last: np.ndarray, # (E,) envs whose episode ends here (time limit, or every agent done)
        acting: np.ndarray, # (E, N) agents whose episode had not ended before this step
    ) -> dict[str, torch.Tensor]:
        "Every agent's transition (E, N, ...): obs and/or image, reward, is_first, is_last, is_terminal, mask, position."
        terminated = np.asarray(columns['terminated'], bool)
        reward = np.asarray(reward, np.float32)
        if self.team_reward:
            reward = np.repeat(reward.sum(-1, keepdims=True), reward.shape[1], -1)
        mask = np.asarray(acting, bool) | np.asarray(is_first, bool)[:, None]
        terminal = terminated & mask & ~np.asarray(is_first, bool)[:, None]
        as_t = lambda x, dtype=torch.bool: torch.as_tensor(np.ascontiguousarray(x), dtype=dtype).unsqueeze(-1)
        trans = {'reward': as_t(reward * mask, torch.float32), 'is_first': as_t(np.broadcast_to(is_first[:, None], mask.shape)),
                 'is_last': as_t((terminal | env_last[:, None]) & mask), 'is_terminal': as_t(terminal),
                 'mask': as_t(mask, torch.float32), 'position': torch.as_tensor(np.asarray(columns['position'], np.int64))}
        if self.obs_dim:
            obs = commnet_features(columns, self.obs_keys, self.grid_size, self.goal)
            trans['obs'] = torch.from_numpy(obs * mask[..., None])
        if self.image_keys:
            trans['image'] = self.image(columns) * torch.from_numpy(mask).to(torch.uint8)[..., None, None, None]
        return trans

    @torch.no_grad()
    def act(self, transitions: dict[str, torch.Tensor], state: dict | None, eval: bool = False):
        "Actions (E, N) for transitions (E, N, ...), and the new latent state (rows: env, then agent)."
        E, N = transitions['mask'].shape[:2]
        flat = {k: v.flatten(0, 1).to(self.device) for k, v in transitions.items()}
        state = self.agent.get_initial_state(E * N) if state is None else state
        action, state = self.agent.act(flat, state, eval=eval)
        return action.float().argmax(-1).view(E, N).cpu(), {k: v.float() for k, v in state.items()}

### Acting

In [ ]:
#| export
@patch
@torch.no_grad()
def get_action(self: DreamerAgents, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) for the info dict of a pool of envs, keeping every agent's latent state."
    columns = {k: torch.as_tensor(info[k])[:, -1].cpu().numpy() for k in self.info_keys()}
    E, N = columns['terminated'].shape
    flush = info.get('_needs_flush')
    flush = np.ones(E, bool) if flush is None else torch.as_tensor(flush).cpu().numpy().astype(bool)
    if self._state is None or len(self._state['deter']) != E * N:
        self._state, flush = None, np.ones(E, bool)
    if flush.any():
        seeds = torch.as_tensor(info['seed'])[:, -1].cpu().numpy() if 'seed' in info else None
        envs = np.flatnonzero(flush)
        self.on_reset(envs, None if seeds is None else [int(seeds[e]) for e in envs])
    acting = ~np.asarray(columns['terminated'], bool)
    trans = self.transitions(columns, np.zeros((E, N)), flush, np.zeros(E, bool), acting)
    action, self._state = self.act(trans, self._state, eval=self.temperature == 0)
    return action


@patch
def reset(self: DreamerAgents):
    "Forget the latent states of the last episodes."
    self._state = None

### Training

`train_dreamer` plays the envs of a `World` without stopping (each env resets with a new seed once
every agent is done or at its time limit), records every agent's transitions, and after
`batch_length + 1` steps of each env, updates the model whenever the replayed agent-steps fall
behind `train_ratio` times the collected ones (r2dreamer's schedule).

In [ ]:
#| export
def train_dreamer(
    model: DreamerAgents,
    world, # A `World`: its envs play without stopping
    steps: int = 100_000, # Env steps (of every env together)
    batch_size: int = 16, # Windows per update (each gives a row per agent)
    batch_length: int = 64,
    train_ratio: float = 512, # Replayed agent-steps per collected agent-step
    capacity: int = 100_000, # Steps kept per env
    seed: int = 0, # Seeds of the episodes: seed, seed + 1, ...
    log_every: int = 1_000, # Env steps between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the env steps and the means since the last call
) -> DreamerAgents:
    """
    Train `model` online, as r2dreamer's online trainer with every agent a row: each step records every
    agent's transition and the action it takes; an env whose episode ended is reset (instead of
    stepped) at the next step. Returns the model in eval mode.
    """
    E, N = world.num_envs, world.num_agents
    replay = SequenceReplay(E, batch_size, batch_length, capacity, model.device)
    next_seed = seed + E
    world.reset(seed=list(range(seed, seed + E)))
    model.on_reset(range(E), list(range(seed, seed + E)))
    state = None
    reward = np.zeros((E, N), np.float32)
    is_first, env_last, acting = np.ones(E, bool), np.zeros(E, bool), np.ones((E, N), bool)
    returns, lengths = np.zeros(E), np.zeros(E, np.int64)
    episodes: list[dict] = []
    totals: dict[str, float] = {}
    updates = count = 0
    model.agent.train()
    for step in range(1, steps // E + 1):
        columns = {k: np.asarray(world.infos[k])[:, -1].copy() for k in model.info_keys()}
        trans = model.transitions(columns, reward, is_first, env_last, acting)
        action, state = model.act(trans, state)
        live = trans['mask'] * torch.from_numpy(~env_last).float().view(E, 1, 1)   # no action after an episode's end
        trans['action'] = F.one_hot(action, model.num_actions).float() * live
        trans['stoch'], trans['deter'] = (state[k].view(E, N, *state[k].shape[1:]).cpu() for k in ('stoch', 'deter'))
        replay.add(trans)
        for e in np.flatnonzero(env_last):                                   # episodes that ended at this step
            episodes.append({'success': bool(columns['terminated'][e].all()), 'return': float(returns[e]),
                             'length': int(lengths[e])})
        acting = acting & ~columns['terminated']                             # agents acting after this step
        if env_last.any():                                                   # r2dreamer: reset instead of stepping
            _, world.infos = world.envs.reset(seed=[next_seed + i for i in range(E)], mask=env_last)
            model.on_reset(np.flatnonzero(env_last), [next_seed + int(e) for e in np.flatnonzero(env_last)])
            next_seed += E
        running = ~env_last
        _, rewards, terms, truncs, world.infos = world.envs.step(action.numpy(), mask=running)
        returns, lengths = np.where(env_last, 0.0, returns + rewards.sum(1)), np.where(env_last, 0, lengths + 1)
        reward = np.where(env_last[:, None], 0.0, rewards).astype(np.float32)
        acting[env_last] = True
        is_first, env_last = env_last.copy(), running & (terms | truncs).all(1)
        collected = step * E * N
        if all(len(s.get('action', ())) > batch_length + 1 for s in replay.streams):
            while updates * batch_size * N * batch_length < collected * train_ratio:
                metrics = model.agent.update(replay)
                updates += 1
                for k in ('loss/dyn', 'loss/rep', 'loss/rew', 'loss/con', 'loss/policy', 'loss/value', 'ret', 'action_entropy'):
                    if k in metrics:
                        totals[k] = totals.get(k, 0.0) + float(metrics[k])
                count += 1
        if log is not None and (step * E) % log_every < E and episodes:
            row = {'env_steps': step * E, 'updates': updates,
                   'success_rate': float(np.mean([e['success'] for e in episodes])),
                   'episode_return': float(np.mean([e['return'] for e in episodes])),
                   'length': float(np.mean([e['length'] for e in episodes]))}
            row.update({k: v / max(count, 1) for k, v in totals.items()})
            log(row)
            episodes, totals, count = [], {}, 0
    model.agent.eval()
    model.reset()
    return model

### Saving and loading

In [ ]:
#| export
@patch
def save(self: DreamerAgents, path: str | Path, **extra: Any) -> Path:
    "Save the configuration, the weights and `extra` (e.g. `env_name`, `env_kwargs`) to `path` (e.g. `dreamer.pt`)."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'config': self.config, 'state_dict': self.state_dict(), **extra}, path)
    return path


@patch(cls_method=True)
def load(cls: DreamerAgents, path: str | Path, map_location: Any = 'cpu') -> DreamerAgents:
    "A model saved with `save`; what it stored besides the model is in `checkpoint`."
    checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    config = dict(checkpoint.pop('config'))
    model = cls(OmegaConf.create(config.pop('config')), **config)
    model.load_state_dict(checkpoint.pop('state_dict'))
    model.checkpoint = checkpoint
    model.agent.eval()
    return model

### Tests

A small model on a small FindGoal: transitions of agents that finished are masked, the replay's
batches go through r2dreamer's update, a short run trains and acts in a `World`, and the model
loads back.

In [ ]:
#| hide
import tempfile
from stable_marl import World
from stable_marl.policy import FeedForwardPolicy

kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=10)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.reset(seed=[0, 1])
columns = {k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}
dim = commnet_features(columns, grid_size=7).shape[-1]
torch.manual_seed(0)
small = dict(deter=128, hidden=32, units=32, discrete=8)
model = DreamerAgents(dreamer_config(**small), dim, 2, grid_size=7)

# an agent that reached its goal: its last step is terminal, then padding
acting = np.array([[True, True], [True, False]])
cols = {**columns, 'terminated': np.array([[False, True], [False, True]])}
t = model.transitions(cols, np.array([[0., 1.], [0., 0.]]), np.zeros(2, bool), np.zeros(2, bool), acting)
test_eq(t['is_terminal'][..., 0], torch.tensor([[False, True], [False, False]]))
test_eq(t['mask'][..., 0], torch.tensor([[1., 1.], [1., 0.]]))
test_eq(t['obs'][1, 1].abs().sum(), torch.tensor(0.))
test_eq(t['reward'][0, 1, 0], torch.tensor(1.))

logs = []
train_dreamer(model, world, steps=80, batch_size=2, batch_length=8, train_ratio=4, log_every=40, log=logs.append)
assert logs and all(np.isfinite(r['loss/dyn']) for r in logs if 'loss/dyn' in r)
world.set_policy(FeedForwardPolicy(model))
test_eq(len(world.evaluate(episodes=2, seed=100)['episode_successes']), 2)

# RGB views: r2dreamer's CNN encoder and decoder (rep_loss 'dreamer', 64 x 64), or the CNN encoder only at the view's size
from stable_marl.wm.dreamer.r2dreamer.networks import ConvDecoder, ConvEncoder
rgb_world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **{**kw, 'tile_size': 8})
rgb_world.reset(seed=[0, 1])
rgb_cols = {k: v[:, -1] for k, v in rgb_world.infos.items() if not k.startswith('_')}
vec = commnet_features(rgb_cols, ['direction'], grid_size=7).shape[-1]
for rep_loss, size in (('dreamer', 64), ('r2dreamer', 56)):
    rgb = DreamerAgents(dreamer_config(rep_loss=rep_loss, depth=4, **small), vec, 2, obs_keys=['direction'], grid_size=7,
                        image_key='pixels', image_size=size)
    test_eq(type(rgb.agent.encoder.encoders[0]), ConvEncoder)
    t = rgb.transitions(rgb_cols, np.zeros((2, 2)), np.ones(2, bool), np.zeros(2, bool), np.ones((2, 2), bool))
    test_eq((t['image'].shape, t['image'].dtype, t['obs'].shape), ((2, 2, size, size, 3), torch.uint8, (2, 2, vec)))
    if rep_loss == 'dreamer':
        assert any(isinstance(m, ConvDecoder) for m in rgb.agent.decoder.modules())
    train_dreamer(rgb, rgb_world, steps=24, batch_size=2, batch_length=8, train_ratio=2)
    rgb_world.set_policy(FeedForwardPolicy(rgb))
    test_eq(len(rgb_world.evaluate(episodes=2, seed=100)['episode_successes']), 2)
test_fail(lambda: DreamerAgents(dreamer_config(**small), 0, 2, obs_keys=[], goal=False))

path = model.save(f'{tempfile.mkdtemp()}/dreamer.pt', env_name='MultiGrid-FindGoal-15x15-v0', env_kwargs=kw)
again = DreamerAgents.load(path)
test_eq(again.checkpoint['env_kwargs'], kw)
trans = model.transitions(columns, np.zeros((2, 2)), np.ones(2, bool), np.zeros(2, bool), np.ones((2, 2), bool))
for k, v in model.state_dict().items():
    test_eq(again.state_dict()[k], v)
same = []
for m in (again, model):                       # acting samples the RSSM's stochastic state: same seed, same actions
    torch.manual_seed(1); same.append(m.act(trans, None, eval=True)[0])
test_eq(same[0], same[1])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()